## 1. Installation and Imports


In [1]:
!pip install python-chess openai transformers sentence-transformers nltk bert-score rouge-score accelerate numpy bitsandbytes -q


In [2]:
import chess
import chess.pgn
import chess.engine
import json
import hashlib
import os
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, asdict
from datetime import datetime
import pickle

# GPU support
import torch

# Detect and configure GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✓ GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"✓ CUDA version: {torch.version.cuda}")
    print(f"✓ GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    # Set memory management
    torch.cuda.empty_cache()
else:
    device = torch.device('cpu')
    print("⚠ GPU not available, using CPU")

# Hugging Face imports
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    AutoModelForSeq2SeqLM,
    pipeline,
    GenerationConfig
)
from transformers import BitsAndBytesConfig
from sentence_transformers import SentenceTransformer

# LLM imports (optional - for OpenAI API)
try:
    from openai import OpenAI
    OPENAI_AVAILABLE = True
except ImportError:
    OPENAI_AVAILABLE = False
    print("OpenAI not available. Will use Hugging Face models.")

# Evaluation metrics
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bert_score

import warnings
warnings.filterwarnings('ignore')

# Fix for Windows asyncio subprocess issue
import sys
if sys.platform == 'win32':
    import asyncio
    # Set Windows event loop policy to support subprocess
    if hasattr(asyncio, 'WindowsProactorEventLoopPolicy'):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        print("✓ Windows event loop policy set for subprocess support")

print(f"\n✓ All imports successful. Using device: {device}")


✓ GPU detected: NVIDIA GeForce RTX 4070 Laptop GPU
✓ CUDA version: 12.1
✓ GPU memory: 8.00 GB


C:\Users\Admin\anaconda3\envs\deepgpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Windows event loop policy set for subprocess support

✓ All imports successful. Using device: cuda


## 4. Stockfish Engine Integration


## 2. Data Structures


In [5]:
# CRITICAL FIX: PovScore API change in python-chess
# This MUST be run after Cell 5 (StockfishEvaluator definition)
# It patches the evaluate_position method to work with newer python-chess versions

import types

def fixed_evaluate_position(self, fen: str) -> Tuple[Optional[float], Optional[int]]:
    """
    Evaluate a position from FEN (fixed for newer python-chess API).
    
    Returns:
        (evaluation_in_centipawns, depth)
    """
    if self.engine is None:
        return None, None
    
    try:
        board = chess.Board(fen)
        info = self.engine.analyse(board, chess.engine.Limit(depth=self.depth))
        
        # Get evaluation score
        score = info.get('score')
        if score is None:
            return None, None
        
        # Convert to centipawns
        # PovScore objects in newer python-chess versions use .relative
        if score.is_mate():
            # Mate in N moves: use large value
            mate_score = score.mate()
            centipawns = 10000 if mate_score > 0 else -10000
        else:
            # Use .relative to get score relative to side to move (in centipawns)
            try:
                centipawns = score.relative
            except AttributeError:
                # Fallback for older python-chess versions
                try:
                    centipawns = score.white() if hasattr(score, 'white') else int(score)
                except:
                    centipawns = 0
        
        depth_used = info.get('depth', self.depth)
        
        return centipawns, depth_used
    except Exception as e:
        print(f"Error evaluating position: {e}")
        return None, None

# Patch the method directly on the class
StockfishEvaluator.evaluate_position = fixed_evaluate_position

# Also patch any existing instances (if system was already created)
import gc
for obj in gc.get_objects():
    if isinstance(obj, StockfishEvaluator):
        obj.evaluate_position = types.MethodType(fixed_evaluate_position, obj)
        print(f"✓ Patched existing StockfishEvaluator instance")

print("✓ Fixed StockfishEvaluator.evaluate_position for newer python-chess API")
print("  Using score.relative instead of score.score()")
print("  IMPORTANT: If errors persist, restart kernel and run cells 1-6 in order.")


NameError: name 'StockfishEvaluator' is not defined

In [ ]:
# Quick fix for Stockfish path issue
# Run this cell if you're getting "Error loading Stockfish" with NotImplementedError

import os
import sys

# Fix Windows asyncio issue
if sys.platform == 'win32':
    import asyncio
    if hasattr(asyncio, 'WindowsProactorEventLoopPolicy'):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        print("✓ Windows event loop policy set")

# Your Stockfish path
STOCKFISH_PATH = r"C:\Users\Admin\Downloads\stockfish\stockfish-windows-x86-64-avx2.exe"

# Verify the path exists
if os.path.exists(STOCKFISH_PATH):
    print(f"✓ Stockfish file found at: {STOCKFISH_PATH}")
    # Test loading
    try:
        test_engine = chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH)
        print("✓ Stockfish engine loaded successfully!")
        test_engine.quit()
    except Exception as e:
        print(f"✗ Error loading Stockfish: {e}")
        print(f"Error type: {type(e).__name__}")
        import traceback
        traceback.print_exc()
else:
    print(f"✗ Stockfish file NOT found at: {STOCKFISH_PATH}")
    print("Please check the path and update STOCKFISH_PATH variable above.")


✓ Windows event loop policy set
✓ Stockfish file found at: C:\Users\Admin\Downloads\stockfish\stockfish-windows-x86-64-avx2.exe
✓ Stockfish engine loaded successfully!


In [4]:
@dataclass
class MoveData:
    """Stores data for a single move."""
    move_number: int
    player: str  # 'white' or 'black'
    san: str  # Standard Algebraic Notation
    fen: str  # FEN after the move
    evaluation: Optional[float] = None  # Centipawns from engine
    depth: Optional[int] = None  # Engine search depth
    commentary: Optional[str] = None  # Generated commentary
    
@dataclass
class GameData:
    """Stores complete game data."""
    game_id: str
    white_player: str
    black_player: str
    result: str
    moves: List[MoveData]
    metadata: Dict
    
@dataclass
class CommentaryMetrics:
    """Quality metrics for commentary."""
    clarity: float  # 0-1
    relevance: float  # 0-1
    factual_accuracy: float  # 0-1
    
@dataclass
class EvaluationScores:
    """Evaluation metrics against reference."""
    bleu: float
    rouge_l: float
    bert_score_f1: float


## 3. PGN Parser


In [47]:
class PGNParser:
    """Parses PGN files into structured game data."""
    
    def __init__(self, pgn_file: str):
        self.pgn_file = pgn_file
        
    def parse_game(self, game_index: int = 0) -> Optional[GameData]:
        """Parse a single game from PGN file."""
        with open(self.pgn_file, 'r', encoding='utf-8', errors='ignore') as f:
            pgn = chess.pgn.read_game(f)
            
            # Skip to desired game
            for _ in range(game_index):
                pgn = chess.pgn.read_game(f)
                if pgn is None:
                    return None
            
            if pgn is None:
                return None
            
            # Extract metadata
            metadata = {
                'event': pgn.headers.get('Event', ''),
                'site': pgn.headers.get('Site', ''),
                'date': pgn.headers.get('UTCDate', ''),
                'time': pgn.headers.get('UTCTime', ''),
                'white_elo': pgn.headers.get('WhiteElo', ''),
                'black_elo': pgn.headers.get('BlackElo', ''),
                'eco': pgn.headers.get('ECO', ''),
                'opening': pgn.headers.get('Opening', ''),
                'time_control': pgn.headers.get('TimeControl', ''),
            }
            
            # Generate game ID
            game_id = hashlib.md5(
                f"{pgn.headers.get('Site', '')}{pgn.headers.get('UTCDate', '')}".encode()
            ).hexdigest()[:12]
            
            # Parse moves
            board = pgn.board()
            moves = []
            move_number = 1
            
            for node in pgn.mainline():
                move = node.move
                san = board.san(move)
                board.push(move)
                fen = board.fen()
                
                # Determine player
                player = 'white' if board.turn == chess.BLACK else 'black'
                
                move_data = MoveData(
                    move_number=move_number,
                    player=player,
                    san=san,
                    fen=fen
                )
                
                moves.append(move_data)
                
                # Increment move number for black moves
                if player == 'black':
                    move_number += 1
            
            return GameData(
                game_id=game_id,
                white_player=pgn.headers.get('White', 'Unknown'),
                black_player=pgn.headers.get('Black', 'Unknown'),
                result=pgn.headers.get('Result', '*'),
                moves=moves,
                metadata=metadata
            )
    
    def parse_all_games(self, limit: Optional[int] = None) -> List[GameData]:
        """Parse all games from PGN file."""
        games = []
        with open(self.pgn_file, 'r', encoding='utf-8', errors='ignore') as f:
            game_count = 0
            while True:
                pgn = chess.pgn.read_game(f)
                if pgn is None:
                    break
                
                # Extract metadata
                metadata = {
                    'event': pgn.headers.get('Event', ''),
                    'site': pgn.headers.get('Site', ''),
                    'date': pgn.headers.get('UTCDate', ''),
                    'time': pgn.headers.get('UTCTime', ''),
                    'white_elo': pgn.headers.get('WhiteElo', ''),
                    'black_elo': pgn.headers.get('BlackElo', ''),
                    'eco': pgn.headers.get('ECO', ''),
                    'opening': pgn.headers.get('Opening', ''),
                    'time_control': pgn.headers.get('TimeControl', ''),
                }
                
                game_id = hashlib.md5(
                    f"{pgn.headers.get('Site', '')}{pgn.headers.get('UTCDate', '')}{game_count}".encode()
                ).hexdigest()[:12]
                
                board = pgn.board()
                moves = []
                move_number = 1
                
                for node in pgn.mainline():
                    move = node.move
                    san = board.san(move)
                    board.push(move)
                    fen = board.fen()
                    
                    player = 'white' if board.turn == chess.BLACK else 'black'
                    
                    move_data = MoveData(
                        move_number=move_number,
                        player=player,
                        san=san,
                        fen=fen
                    )
                    
                    moves.append(move_data)
                    
                    if player == 'black':
                        move_number += 1
                
                games.append(GameData(
                    game_id=game_id,
                    white_player=pgn.headers.get('White', 'Unknown'),
                    black_player=pgn.headers.get('Black', 'Unknown'),
                    result=pgn.headers.get('Result', '*'),
                    moves=moves,
                    metadata=metadata
                ))
                
                game_count += 1
                if limit and game_count >= limit:
                    break
        
        return games


In [48]:
class StockfishEvaluator:
    """Evaluates chess positions using Stockfish engine."""
    
    def __init__(self, stockfish_path: Optional[str] = None, depth: int = 15):
        """
        Initialize Stockfish evaluator.
        
        Args:
            stockfish_path: Path to Stockfish executable. If None, tries to find it.
            depth: Search depth for evaluations
        """
        self.depth = depth
        self.engine = None
        
        # Try to find Stockfish
        if stockfish_path is None:
            # Common paths
            possible_paths = [
                'stockfish',  # In PATH
                'stockfish.exe',  # Windows
                '/usr/local/bin/stockfish',  # macOS
                '/usr/bin/stockfish',  # Linux
                'C:\\Stockfish\\stockfish.exe',  # Windows default
            ]
            
            for path in possible_paths:
                try:
                    self.engine = chess.engine.SimpleEngine.popen_uci(path)
                    print(f"Stockfish found at: {path}")
                    break
                except:
                    continue
            
            if self.engine is None:
                print("Warning: Stockfish not found. Install it or provide path.")
                print("For Windows: https://stockfishchess.org/download/")
        else:
            try:
                self.engine = chess.engine.SimpleEngine.popen_uci(stockfish_path)
            except Exception as e:
                print(f"Error loading Stockfish: {e}")
    
    def evaluate_position(self, fen: str) -> Tuple[Optional[float], Optional[int]]:
        """
        Evaluate a position from FEN.
        
        Returns:
            (evaluation_in_centipawns, depth)
        """
        if self.engine is None:
            return None, None
        
        try:
            board = chess.Board(fen)
            info = self.engine.analyse(board, chess.engine.Limit(depth=self.depth))
            
            # Get evaluation score
            score = info.get('score')
            if score is None:
                return None, None
            
            # Convert to centipawns
            if score.is_mate():
                # Mate in N moves: use large value
                mate_score = score.mate()
                centipawns = 10000 if mate_score > 0 else -10000
            else:
                centipawns = score.score()
            
            depth_used = info.get('depth', self.depth)
            
            return centipawns, depth_used
        except Exception as e:
            print(f"Error evaluating position: {e}")
            return None, None
    
    def evaluate_game(self, game: GameData) -> GameData:
        """Evaluate all positions in a game."""
        if self.engine is None:
            print("Stockfish not available. Skipping evaluations.")
            return game
        
        for move_data in game.moves:
            if move_data.evaluation is None:
                eval_score, depth = self.evaluate_position(move_data.fen)
                move_data.evaluation = eval_score
                move_data.depth = depth
        
        return game
    
    def close(self):
        """Close the engine."""
        if self.engine:
            self.engine.quit()
    
    def __enter__(self):
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        self.close()


In [49]:
import re

class CommentaryGenerator:
    """Generates natural language commentary using Hugging Face models or OpenAI API."""
    
    def __init__(self, 
                 model_name: str = "google/flan-t5-base",
                 use_openai: bool = False,
                 api_key: Optional[str] = None,
                 openai_model: str = "gpt-4o-mini",
                 device: Optional[torch.device] = None,
                 use_quantization: bool = False):
        """
        Initialize commentary generator with Hugging Face model.
        
        Args:
            model_name: Hugging Face model name (e.g., "google/flan-t5-base", "gpt2", "microsoft/DialoGPT-medium")
            use_openai: If True, use OpenAI API instead of Hugging Face
            api_key: OpenAI API key (if using OpenAI)
            openai_model: OpenAI model name
            device: PyTorch device (auto-detects if None)
            use_quantization: Use 8-bit quantization for memory efficiency
        """
        self.model_name = model_name
        self.use_openai = use_openai
        self.device = device if device is not None else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        # Initialize OpenAI client if requested
        self.openai_client = None
        if use_openai and OPENAI_AVAILABLE:
            api_key = api_key or os.getenv('OPENAI_API_KEY')
            if api_key:
                self.openai_client = OpenAI(api_key=api_key)
                self.openai_model = openai_model
                print(f"✓ Using OpenAI API with model: {openai_model}")
            else:
                print("⚠ No OpenAI API key provided. Falling back to Hugging Face model.")
                self.use_openai = False
        
        # Initialize Hugging Face model
        if not self.use_openai:
            print(f"Loading Hugging Face model: {model_name}")
            print(f"Using device: {self.device}")
            
            try:
                # Check if model is seq2seq (T5, FLAN) or causal (GPT)
                if "t5" in model_name.lower() or "flan" in model_name.lower():
                    self.model_type = "seq2seq"
                    self.tokenizer = AutoTokenizer.from_pretrained(model_name)
                    
                    if use_quantization and self.device.type == 'cuda':
                        # Use 8-bit quantization for memory efficiency
                        quantization_config = BitsAndBytesConfig(
                            load_in_8bit=True,
                            device_map="auto"
                        )
                        self.model = AutoModelForSeq2SeqLM.from_pretrained(
                            model_name,
                            quantization_config=quantization_config,
                            device_map="auto",
                            torch_dtype=torch.float16
                        )
                    else:
                        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
                        self.model = self.model.to(self.device)
                        if self.device.type == 'cuda':
                            self.model = self.model.half()  # Use FP16 for efficiency
                    
                    self.model.eval()
                    print(f"✓ Loaded {model_name} (seq2seq) on {self.device}")
                    
                else:
                    # Causal LM (GPT-2, GPT-Neo, etc.)
                    self.model_type = "causal"
                    self.tokenizer = AutoTokenizer.from_pretrained(model_name)
                    
                    # Add padding token if not present
                    if self.tokenizer.pad_token is None:
                        self.tokenizer.pad_token = self.tokenizer.eos_token
                    
                    if use_quantization and self.device.type == 'cuda':
                        quantization_config = BitsAndBytesConfig(
                            load_in_8bit=True,
                            device_map="auto"
                        )
                        self.model = AutoModelForCausalLM.from_pretrained(
                            model_name,
                            quantization_config=quantization_config,
                            device_map="auto",
                            torch_dtype=torch.float16
                        )
                    else:
                        self.model = AutoModelForCausalLM.from_pretrained(model_name)
                        self.model = self.model.to(self.device)
                        if self.device.type == 'cuda':
                            self.model = self.model.half()
                    
                    self.model.eval()
                    print(f"✓ Loaded {model_name} (causal) on {self.device}")
                    
            except Exception as e:
                print(f"Error loading Hugging Face model: {e}")
                print("Falling back to rule-based commentary.")
                self.model = None
                self.tokenizer = None
        else:
            self.model = None
            self.tokenizer = None
    
    def _get_system_prompt(self) -> str:
        """Get system prompt for commentary generation."""
        return """You are an expert chess commentator. Generate clear, concise, and accurate 
commentary for chess moves. Focus on:
1. Key tactical and positional ideas
2. Move quality and evaluation
3. Threats and opportunities
4. Strategic plans

Keep commentary to 1-2 sentences. Be factual and avoid speculation."""
    
    def _generate_with_openai(self, prompt: str) -> str:
        """Generate commentary using OpenAI API."""
        try:
            response = self.openai_client.chat.completions.create(
                model=self.openai_model,
                messages=[
                    {"role": "system", "content": self._get_system_prompt()},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.7,
                max_tokens=200
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            print(f"Error with OpenAI API: {e}")
            return self._fallback_commentary(prompt)
    
    def _generate_with_hf(self, prompt: str) -> str:
        """Generate commentary using Hugging Face model."""
        if self.model is None or self.tokenizer is None:
            return self._fallback_commentary(prompt)
        
        try:
            # Build full prompt with instruction
            if self.model_type == "seq2seq":
                # For T5/FLAN models, use instruction format
                full_prompt = f"Comment on this chess move: {prompt}"
                inputs = self.tokenizer(
                    full_prompt,
                    return_tensors="pt",
                    max_length=512,
                    truncation=True,
                    padding=True
                ).to(self.device)
                
                with torch.no_grad():
                    outputs = self.model.generate(
                        **inputs,
                        max_length=150,
                        min_length=20,
                        num_beams=4,
                        early_stopping=True,
                        temperature=0.7,
                        do_sample=True,
                        pad_token_id=self.tokenizer.pad_token_id
                    )
                
                commentary = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
                
            else:
                # For causal models (GPT-2, etc.)
                instruction = "Comment on this chess move in 1-2 sentences: "
                full_prompt = instruction + prompt
                inputs = self.tokenizer(
                    full_prompt,
                    return_tensors="pt",
                    max_length=256,
                    truncation=True
                ).to(self.device)
                
                with torch.no_grad():
                    outputs = self.model.generate(
                        **inputs,
                        max_new_tokens=100,
                        min_length=20,
                        temperature=0.7,
                        do_sample=True,
                        pad_token_id=self.tokenizer.pad_token_id,
                        eos_token_id=self.tokenizer.eos_token_id,
                        repetition_penalty=1.2
                    )
                
                # Decode only the new tokens
                generated_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
                # Remove the input prompt from output
                commentary = generated_text[len(full_prompt):].strip()
            
            # Clean up commentary
            commentary = commentary.strip()
            if not commentary or len(commentary) < 10:
                return self._fallback_commentary(prompt)
            
            # Limit to reasonable length
            sentences = commentary.split('.')
            if len(sentences) > 2:
                commentary = '. '.join(sentences[:2]) + '.'
            
            return commentary
            
        except Exception as e:
            print(f"Error generating with Hugging Face model: {e}")
            return self._fallback_commentary(prompt)
    
    def _fallback_commentary(self, prompt: str) -> str:
        """Fallback commentary when models are unavailable."""
        # Simple rule-based commentary
        if "evaluation" in prompt.lower():
            eval_match = re.search(r'evaluation[\s:]+([+-]?\d+)', prompt, re.IGNORECASE)
            if eval_match:
                eval_val = int(eval_match.group(1))
                if eval_val > 200:
                    return "White has a strong advantage with this move."
                elif eval_val < -200:
                    return "Black has a strong advantage with this move."
                elif abs(eval_val) < 50:
                    return "The position remains roughly equal after this move."
        
        return "A standard move in this position."
    
    def generate_commentary(self, move_data: MoveData, previous_move: Optional[MoveData] = None, 
                          game_context: Optional[Dict] = None) -> str:
        """
        Generate commentary for a single move.
        
        Args:
            move_data: Current move data
            previous_move: Previous move (for context)
            game_context: Additional game context
        """
        # Build prompt
        prompt_parts = [
            f"Move {move_data.move_number}: {move_data.player.capitalize()} plays {move_data.san}",
        ]
        
        if move_data.evaluation is not None:
            eval_cp = move_data.evaluation
            if abs(eval_cp) > 9000:
                prompt_parts.append(f"Evaluation: Mate in favor of {'White' if eval_cp > 0 else 'Black'}")
            else:
                prompt_parts.append(f"Evaluation: {eval_cp/100:.2f} pawns ({'White' if eval_cp > 0 else 'Black'} advantage)")
        
        if previous_move:
            prompt_parts.append(f"Previous move: {previous_move.san}")
        
        if game_context:
            if game_context.get('opening'):
                prompt_parts.append(f"Opening: {game_context['opening']}")
        
        prompt = ". ".join(prompt_parts)
        
        # Generate commentary
        if self.use_openai and self.openai_client:
            commentary = self._generate_with_openai(prompt)
        else:
            commentary = self._generate_with_hf(prompt)
        
        # Store commentary
        move_data.commentary = commentary
        
        return commentary
    
    def generate_game_commentary(self, game: GameData) -> GameData:
        """Generate commentary for all moves in a game."""
        previous_move = None
        
        for move_data in game.moves:
            if move_data.commentary is None:
                self.generate_commentary(
                    move_data, 
                    previous_move, 
                    game_context=game.metadata
                )
            previous_move = move_data
        
        return game
    
    def __del__(self):
        """Cleanup model from memory."""
        if hasattr(self, 'model') and self.model is not None:
            del self.model
            if self.device.type == 'cuda':
                torch.cuda.empty_cache()


## 6. Quality Metrics (Clarity, Relevance, Factual Accuracy)


In [50]:
class QualityMetrics:
    """Evaluates commentary quality metrics using rule-based and semantic methods."""
    
    def __init__(self, device: Optional[torch.device] = None, use_semantic: bool = True):
        """
        Initialize quality metrics evaluator.
        
        Args:
            device: PyTorch device for sentence transformers
            use_semantic: Use sentence transformers for semantic evaluation
        """
        self.device = device if device is not None else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.use_semantic = use_semantic
        self.sentence_model = None
        
        if use_semantic:
            try:
                # Use a lightweight sentence transformer model
                model_name = "all-MiniLM-L6-v2"  # Fast and efficient
                print(f"Loading sentence transformer model: {model_name} for semantic evaluation")
                self.sentence_model = SentenceTransformer(model_name, device=str(self.device))
                print(f"✓ Sentence transformer loaded on {self.device}")
            except Exception as e:
                print(f"⚠ Could not load sentence transformer: {e}. Using rule-based only.")
                self.use_semantic = False
    
    def calculate_clarity(self, commentary: str) -> float:
        """
        Calculate clarity score (0-1).
        Based on sentence length, readability, and structure.
        """
        if not commentary:
            return 0.0
        
        sentences = commentary.split('.')
        sentences = [s.strip() for s in sentences if s.strip()]
        
        if not sentences:
            return 0.0
        
        # Average sentence length (words)
        avg_length = sum(len(s.split()) for s in sentences) / len(sentences)
        
        # Optimal length is 10-20 words
        if 10 <= avg_length <= 20:
            length_score = 1.0
        elif 5 <= avg_length < 10 or 20 < avg_length <= 30:
            length_score = 0.7
        else:
            length_score = 0.4
        
        # Check for proper capitalization
        capitalization_score = 1.0 if commentary[0].isupper() else 0.5
        
        # Check for proper punctuation
        punctuation_score = 1.0 if commentary[-1] in '.!?' else 0.7
        
        return (length_score * 0.5 + capitalization_score * 0.25 + punctuation_score * 0.25)
    
    def calculate_relevance(self, commentary: str, move_data: MoveData) -> float:
        """
        Calculate relevance score (0-1).
        Checks if commentary mentions the move, position, or evaluation.
        Uses semantic similarity if available.
        """
        if not commentary:
            return 0.0
        
        commentary_lower = commentary.lower()
        san_lower = move_data.san.lower()
        
        # Rule-based scoring
        move_mentioned = False
        if san_lower in commentary_lower:
            move_mentioned = True
        else:
            pieces = ['pawn', 'knight', 'bishop', 'rook', 'queen', 'king']
            for piece in pieces:
                if piece in commentary_lower and any(p in san_lower for p in ['n', 'b', 'r', 'q', 'k']):
                    move_mentioned = True
                    break
        
        chess_terms = ['move', 'position', 'advantage', 'threat', 'attack', 'defense', 
                      'tactic', 'strategy', 'check', 'mate', 'capture', 'castling']
        chess_terms_found = sum(1 for term in chess_terms if term in commentary_lower)
        
        eval_mentioned = False
        if move_data.evaluation is not None:
            eval_terms = ['advantage', 'better', 'stronger', 'winning', 'losing', 'equal']
            eval_mentioned = any(term in commentary_lower for term in eval_terms)
        
        rule_score = 0.0
        if move_mentioned:
            rule_score += 0.4
        if chess_terms_found > 0:
            rule_score += min(0.4, chess_terms_found * 0.1)
        if eval_mentioned or move_data.evaluation is None:
            rule_score += 0.2
        
        # Add semantic similarity if available
        semantic_score = 0.0
        if self.use_semantic and self.sentence_model is not None:
            try:
                # Create expected commentary based on move
                expected_text = f"Chess move {move_data.san} in position"
                if move_data.evaluation is not None:
                    eval_cp = move_data.evaluation
                    if abs(eval_cp) > 200:
                        expected_text += f" with {'strong advantage' if eval_cp > 0 else 'disadvantage'}"
                
                # Calculate semantic similarity
                embeddings = self.sentence_model.encode([commentary, expected_text], convert_to_tensor=True)
                similarity = torch.nn.functional.cosine_similarity(embeddings[0:1], embeddings[1:2]).item()
                semantic_score = max(0, similarity) * 0.3  # Weight semantic score
            except Exception as e:
                pass  # Fall back to rule-based only
        
        return min(1.0, rule_score + semantic_score)
    
    def calculate_factual_accuracy(self, commentary: str, move_data: MoveData, 
                                  previous_move: Optional[MoveData] = None) -> float:
        """
        Calculate factual accuracy score (0-1).
        Checks if commentary correctly describes the move and position.
        """
        if not commentary:
            return 0.0
        
        commentary_lower = commentary.lower()
        score = 1.0
        
        # Check if evaluation is consistent with commentary
        if move_data.evaluation is not None:
            eval_cp = move_data.evaluation
            
            # Check for contradictions
            if eval_cp > 200:
                # White advantage - should not say black is better
                if any(term in commentary_lower for term in ['black advantage', 'black better', 'black winning']):
                    score -= 0.3
            elif eval_cp < -200:
                # Black advantage - should not say white is better
                if any(term in commentary_lower for term in ['white advantage', 'white better', 'white winning']):
                    score -= 0.3
            elif abs(eval_cp) < 50:
                # Equal position - should not strongly favor either side
                if any(term in commentary_lower for term in ['strong advantage', 'winning', 'crushing']):
                    score -= 0.2
        
        # Check for impossible moves (basic sanity checks)
        # This is simplified - full validation would require position analysis
        
        return max(0.0, min(1.0, score))
    
    def evaluate_commentary(self, move_data: MoveData, previous_move: Optional[MoveData] = None) -> CommentaryMetrics:
        """Evaluate all quality metrics for a move's commentary."""
        if not move_data.commentary:
            return CommentaryMetrics(clarity=0.0, relevance=0.0, factual_accuracy=0.0)
        
        clarity = self.calculate_clarity(move_data.commentary)
        relevance = self.calculate_relevance(move_data.commentary, move_data)
        factual_accuracy = self.calculate_factual_accuracy(
            move_data.commentary, move_data, previous_move
        )
        
        return CommentaryMetrics(
            clarity=clarity,
            relevance=relevance,
            factual_accuracy=factual_accuracy
        )


## 7. Caching System


In [51]:
class CommentaryCache:
    """Caches commentary and evaluations to avoid redundant computation."""
    
    def __init__(self, cache_dir: str = "commentary_cache"):
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(exist_ok=True)
        
        # In-memory cache for faster access
        self.memory_cache = {}
    
    def _get_cache_key(self, fen: str, san: str, evaluation: Optional[float] = None) -> str:
        """Generate cache key from position and move."""
        key_data = f"{fen}|{san}|{evaluation}"
        return hashlib.md5(key_data.encode()).hexdigest()
    
    def get_commentary(self, fen: str, san: str, evaluation: Optional[float] = None) -> Optional[str]:
        """Retrieve cached commentary."""
        cache_key = self._get_cache_key(fen, san, evaluation)
        
        # Check memory cache first
        if cache_key in self.memory_cache:
            return self.memory_cache[cache_key]
        
        # Check disk cache
        cache_file = self.cache_dir / f"{cache_key}.pkl"
        if cache_file.exists():
            try:
                with open(cache_file, 'rb') as f:
                    data = pickle.load(f)
                    commentary = data.get('commentary')
                    # Store in memory cache
                    self.memory_cache[cache_key] = commentary
                    return commentary
            except Exception as e:
                print(f"Error reading cache: {e}")
        
        return None
    
    def store_commentary(self, fen: str, san: str, commentary: str, 
                        evaluation: Optional[float] = None):
        """Store commentary in cache."""
        cache_key = self._get_cache_key(fen, san, evaluation)
        
        # Store in memory cache
        self.memory_cache[cache_key] = commentary
        
        # Store on disk
        cache_file = self.cache_dir / f"{cache_key}.pkl"
        try:
            with open(cache_file, 'wb') as f:
                pickle.dump({'commentary': commentary, 'fen': fen, 'san': san}, f)
        except Exception as e:
            print(f"Error writing cache: {e}")
    
    def get_evaluation(self, fen: str) -> Optional[Tuple[float, int]]:
        """Retrieve cached evaluation."""
        cache_key = hashlib.md5(f"eval|{fen}".encode()).hexdigest()
        cache_file = self.cache_dir / f"eval_{cache_key}.pkl"
        
        if cache_file.exists():
            try:
                with open(cache_file, 'rb') as f:
                    return pickle.load(f)
            except Exception as e:
                print(f"Error reading evaluation cache: {e}")
        
        return None
    
    def store_evaluation(self, fen: str, evaluation: float, depth: int):
        """Store evaluation in cache."""
        cache_key = hashlib.md5(f"eval|{fen}".encode()).hexdigest()
        cache_file = self.cache_dir / f"eval_{cache_key}.pkl"
        
        try:
            with open(cache_file, 'wb') as f:
                pickle.dump((evaluation, depth), f)
        except Exception as e:
            print(f"Error writing evaluation cache: {e}")
    
    def clear_cache(self):
        """Clear all cached data."""
        self.memory_cache.clear()
        for file in self.cache_dir.glob("*.pkl"):
            try:
                file.unlink()
            except Exception as e:
                print(f"Error deleting cache file {file}: {e}")


## 8. Evaluation Metrics (BLEU, ROUGE-L, BERTScore)


In [52]:
class CommentaryEvaluator:
    """Evaluates generated commentary against reference commentary."""
    
    def __init__(self, device: Optional[torch.device] = None):
        """
        Initialize evaluator.
        
        Args:
            device: PyTorch device (cuda/cpu). If None, auto-detects.
        """
        self.rouge_scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
        self.smoothing = SmoothingFunction().method1
        
        # Set device for GPU acceleration
        if device is None:
            self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        else:
            self.device = device
        
        if self.device.type == 'cuda':
            print(f"✓ Using GPU for BERTScore evaluation")
        else:
            print("⚠ Using CPU for BERTScore evaluation")
    
    def calculate_bleu(self, generated: str, reference: str) -> float:
        """Calculate BLEU score."""
        if not generated or not reference:
            return 0.0
        
        # Tokenize
        gen_tokens = generated.lower().split()
        ref_tokens = reference.lower().split()
        
        if not gen_tokens or not ref_tokens:
            return 0.0
        
        try:
            score = sentence_bleu(
                [ref_tokens], 
                gen_tokens,
                smoothing_function=self.smoothing
            )
            return score
        except Exception as e:
            print(f"Error calculating BLEU: {e}")
            return 0.0
    
    def calculate_rouge_l(self, generated: str, reference: str) -> float:
        """Calculate ROUGE-L score."""
        if not generated or not reference:
            return 0.0
        
        try:
            scores = self.rouge_scorer.score(reference, generated)
            return scores['rougeL'].fmeasure
        except Exception as e:
            print(f"Error calculating ROUGE-L: {e}")
            return 0.0
    
    def calculate_bert_score(self, generated: str, reference: str) -> float:
        """Calculate BERTScore F1 using GPU if available."""
        if not generated or not reference:
            return 0.0
        
        try:
            # BERTScore expects lists and can use GPU via device parameter
            # Convert device to string format expected by bert_score
            device_str = str(self.device) if self.device.type == 'cuda' else 'cpu'
            P, R, F1 = bert_score(
                [generated], 
                [reference], 
                lang='en', 
                verbose=False,
                device=device_str,
                batch_size=1 if self.device.type == 'cuda' else 8  # Smaller batch for GPU memory
            )
            return F1.item()
        except Exception as e:
            print(f"Error calculating BERTScore: {e}")
            return 0.0
    
    def evaluate(self, generated: str, reference: str) -> EvaluationScores:
        """Calculate all evaluation metrics."""
        bleu = self.calculate_bleu(generated, reference)
        rouge_l = self.calculate_rouge_l(generated, reference)
        bert_f1 = self.calculate_bert_score(generated, reference)
        
        return EvaluationScores(
            bleu=bleu,
            rouge_l=rouge_l,
            bert_score_f1=bert_f1
        )
    
    def evaluate_batch(self, generated_list: List[str], reference_list: List[str]) -> Dict[str, float]:
        """Evaluate a batch of commentaries and return average scores."""
        if len(generated_list) != len(reference_list):
            raise ValueError("Generated and reference lists must have same length")
        
        # For BERTScore, batch processing is more efficient
        try:
            device_str = str(self.device) if self.device.type == 'cuda' else 'cpu'
            batch_size = 4 if self.device.type == 'cuda' else 16
            
            # Calculate BERTScore for entire batch at once (more efficient)
            P, R, F1 = bert_score(
                generated_list,
                reference_list,
                lang='en',
                verbose=False,
                device=device_str,
                batch_size=batch_size
            )
            bert_scores = F1.tolist()
        except Exception as e:
            print(f"Error in batch BERTScore: {e}, falling back to individual calculation")
            bert_scores = [self.calculate_bert_score(gen, ref) for gen, ref in zip(generated_list, reference_list)]
        
        # Calculate BLEU and ROUGE-L for each pair
        scores = []
        for i, (gen, ref) in enumerate(zip(generated_list, reference_list)):
            bleu = self.calculate_bleu(gen, ref)
            rouge_l = self.calculate_rouge_l(gen, ref)
            scores.append(EvaluationScores(
                bleu=bleu,
                rouge_l=rouge_l,
                bert_score_f1=bert_scores[i] if i < len(bert_scores) else 0.0
            ))
        
        return {
            'bleu': sum(s.bleu for s in scores) / len(scores),
            'rouge_l': sum(s.rouge_l for s in scores) / len(scores),
            'bert_score_f1': sum(s.bert_score_f1 for s in scores) / len(scores)
        }


## 9. Main Pipeline


In [53]:
class ChessCommentarySystem:
    """Main system for generating chess commentary."""
    
    def __init__(self, 
                 stockfish_path: Optional[str] = None,
                 stockfish_depth: int = 15,
                 llm_api_key: Optional[str] = None,
                 llm_model: str = "gpt-4o-mini",
                 cache_dir: str = "commentary_cache",
                 device: Optional[torch.device] = None):
        """
        Initialize the commentary system.
        
        Args:
            stockfish_path: Path to Stockfish executable
            stockfish_depth: Search depth for Stockfish
            llm_api_key: OpenAI API key
            llm_model: LLM model to use
            cache_dir: Directory for caching results
            device: PyTorch device for GPU acceleration (auto-detects if None)
        """
        # Auto-detect device if not provided
        if device is None:
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        self.device = device
        self.evaluator = StockfishEvaluator(stockfish_path, stockfish_depth)
        # Initialize commentary generator with Hugging Face model
        self.commentary_gen = CommentaryGenerator(
            model_name=llm_model if not llm_model.startswith('gpt') else "google/flan-t5-base",
            use_openai=llm_model.startswith('gpt') and llm_api_key is not None,
            api_key=llm_api_key,
            openai_model=llm_model if llm_model.startswith('gpt') else "gpt-4o-mini",
            device=self.device,
            use_quantization=False  # Set to True for memory-constrained GPUs
        )
        self.cache = CommentaryCache(cache_dir)
        self.quality_metrics = QualityMetrics(device=self.device)
        # Use GPU if available for BERTScore evaluation
        self.evaluator_metrics = CommentaryEvaluator(device=self.device)
    
    def process_game(self, game: GameData, use_cache: bool = True) -> GameData:
        """
        Process a game: evaluate positions and generate commentary.
        
        Args:
            game: GameData object
            use_cache: Whether to use cached results
        """
        previous_move = None
        
        for move_data in game.moves:
            # Get evaluation (with caching)
            if move_data.evaluation is None:
                if use_cache:
                    cached_eval = self.cache.get_evaluation(move_data.fen)
                    if cached_eval:
                        move_data.evaluation, move_data.depth = cached_eval
                    else:
                        eval_score, depth = self.evaluator.evaluate_position(move_data.fen)
                        move_data.evaluation = eval_score
                        move_data.depth = depth
                        if eval_score is not None:
                            self.cache.store_evaluation(move_data.fen, eval_score, depth)
                else:
                    eval_score, depth = self.evaluator.evaluate_position(move_data.fen)
                    move_data.evaluation = eval_score
                    move_data.depth = depth
            
            # Generate commentary (with caching)
            if move_data.commentary is None:
                if use_cache:
                    cached_commentary = self.cache.get_commentary(
                        move_data.fen, move_data.san, move_data.evaluation
                    )
                    if cached_commentary:
                        move_data.commentary = cached_commentary
                    else:
                        commentary = self.commentary_gen.generate_commentary(
                            move_data, previous_move, game.metadata
                        )
                        self.cache.store_commentary(
                            move_data.fen, move_data.san, commentary, move_data.evaluation
                        )
                else:
                    commentary = self.commentary_gen.generate_commentary(
                        move_data, previous_move, game.metadata
                    )
            
            previous_move = move_data
        
        return game
    
    def evaluate_quality(self, game: GameData) -> List[CommentaryMetrics]:
        """Evaluate quality metrics for all moves in a game."""
        metrics = []
        previous_move = None
        
        for move_data in game.moves:
            metric = self.quality_metrics.evaluate_commentary(move_data, previous_move)
            metrics.append(metric)
            previous_move = move_data
        
        return metrics
    
    def evaluate_against_reference(self, generated: List[str], reference: List[str]) -> Dict[str, float]:
        """Evaluate generated commentary against reference."""
        return self.evaluator_metrics.evaluate_batch(generated, reference)
    
    def close(self):
        """Close resources and free GPU memory."""
        self.evaluator.close()
        # Clean up models
        if hasattr(self.commentary_gen, 'model') and self.commentary_gen.model is not None:
            del self.commentary_gen.model
        if hasattr(self.quality_metrics, 'sentence_model') and self.quality_metrics.sentence_model is not None:
            del self.quality_metrics.sentence_model
        if self.device.type == 'cuda':
            torch.cuda.empty_cache()
            print("✓ GPU memory cleared")
    
    def __enter__(self):
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        self.close()


## 10. Example Usage


In [54]:
# Initialize system
# Note: For Hugging Face models, specify model name (e.g., "google/flan-t5-base", "gpt2")
# For OpenAI API, set OPENAI_API_KEY and use model like "gpt-4o-mini"
# GPU will be auto-detected and used for all models

# Option 1: Use Hugging Face model (default, no API key needed)
system = ChessCommentarySystem(
    stockfish_path="C:\\Users\\Admin\\Downloads\\stockfish\\stockfish-windows-x86-64-avx2.exe",
    stockfish_depth=15,
    llm_api_key=None,  # Not needed for Hugging Face models
    llm_model="google/flan-t5-base",  # Hugging Face model name
    cache_dir="commentary_cache",
    device=device  # Uses GPU if available (from imports cell)
)

# Option 2: Use OpenAI API (uncomment to use)
# system = ChessCommentarySystem(
#     stockfish_path=None,
#     stockfish_depth=15,
#     llm_api_key=os.getenv('OPENAI_API_KEY'),  # Set your API key
#     llm_model="gpt-4o-mini",  # OpenAI model
#     cache_dir="commentary_cache",
#     device=device
# )


Loading Hugging Face model: google/flan-t5-base
Using device: cuda
✓ Loaded google/flan-t5-base (seq2seq) on cuda
Loading sentence transformer model: all-MiniLM-L6-v2 for semantic evaluation
✓ Sentence transformer loaded on cuda
✓ Using GPU for BERTScore evaluation


In [55]:
# Parse a game from PGN
parser = PGNParser("C:\\Users\\Admin\\Desktop\\AIP_DeepChessIQ\\NLP_LLM\\games.pgn")
game = parser.parse_game(game_index=0)  # First game

if game:
    print(f"Game ID: {game.game_id}")
    print(f"White: {game.white_player} vs Black: {game.black_player}")
    print(f"Result: {game.result}")
    print(f"Number of moves: {len(game.moves)}")
    print(f"Opening: {game.metadata.get('opening', 'Unknown')}")
    print("\nFirst 5 moves:")
    for move in game.moves[:5]:
        print(f"  {move.move_number}. {move.player}: {move.san}")


Game ID: 3073e53b9ef1
White: BFG9k vs Black: mamalak
Result: 1-0
Number of moves: 25
Opening: French Defense: Normal Variation

First 5 moves:
  1. white: e4
  1. black: e6
  2. white: d4
  2. black: b6
  3. white: a3


In [56]:
# Process game: get evaluations and generate commentary
if game:
    processed_game = system.process_game(game, use_cache=True)
    
    print("\n=== Game Commentary ===")
    print(f"{processed_game.white_player} vs {processed_game.black_player}")
    print(f"Result: {processed_game.result}\n")
    
    for move_data in processed_game.moves[:10]:  # Show first 10 moves
        eval_str = "N/A"
        if move_data.evaluation is not None:
            if abs(move_data.evaluation) > 9000:
                eval_str = "Mate"
            else:
                eval_str = f"{move_data.evaluation/100:.2f} cp"
        
        print(f"Move {move_data.move_number} ({move_data.player}): {move_data.san}")
        print(f"  Evaluation: {eval_str}")
        print(f"  Commentary: {move_data.commentary}")
        print()


Error evaluating position: 'PovScore' object has no attribute 'score'
Error evaluating position: 'PovScore' object has no attribute 'score'
Error evaluating position: 'PovScore' object has no attribute 'score'
Error evaluating position: 'PovScore' object has no attribute 'score'
Error evaluating position: 'PovScore' object has no attribute 'score'
Error evaluating position: 'PovScore' object has no attribute 'score'
Error evaluating position: 'PovScore' object has no attribute 'score'
Error evaluating position: 'PovScore' object has no attribute 'score'
Error evaluating position: 'PovScore' object has no attribute 'score'
Error evaluating position: 'PovScore' object has no attribute 'score'
Error evaluating position: 'PovScore' object has no attribute 'score'
Error evaluating position: 'PovScore' object has no attribute 'score'
Error evaluating position: 'PovScore' object has no attribute 'score'
Error evaluating position: 'PovScore' object has no attribute 'score'
Error evaluating pos

In [ ]:
# Evaluate quality metrics
if game:
    quality_metrics = system.evaluate_quality(processed_game)
    
    print("\n=== Quality Metrics ===")
    avg_clarity = sum(m.clarity for m in quality_metrics) / len(quality_metrics)
    avg_relevance = sum(m.relevance for m in quality_metrics) / len(quality_metrics)
    avg_accuracy = sum(m.factual_accuracy for m in quality_metrics) / len(quality_metrics)
    
    print(f"Average Clarity: {avg_clarity:.3f}")
    print(f"Average Relevance: {avg_relevance:.3f}")
    print(f"Average Factual Accuracy: {avg_accuracy:.3f}")
    print(f"Overall Quality Score: {(avg_clarity + avg_relevance + avg_accuracy) / 3:.3f}")



=== Quality Metrics ===
Average Clarity: 0.718
Average Relevance: 0.466
Average Factual Accuracy: 1.000
Overall Quality Score: 0.728


In [ ]:
# Example: Evaluate against reference commentary
# (In practice, you would have reference commentary from experts)

# Example reference commentaries for first 5 moves
reference_commentaries = [
    "White opens with the king's pawn, a common and solid first move.",
    "Black responds with e6, preparing the French Defense.",
    "White advances the d-pawn, establishing central control.",
    "Black plays b6, developing the bishop to b7.",
    "White plays a3, a prophylactic move."
]

# Get generated commentaries
generated_commentaries = [move.commentary for move in processed_game.moves[:5] if move.commentary]

if len(generated_commentaries) == len(reference_commentaries):
    eval_scores = system.evaluate_against_reference(generated_commentaries, reference_commentaries)
    
    print("\n=== Evaluation Metrics vs Reference ===")
    print(f"BLEU Score: {eval_scores['bleu']:.4f}")
    print(f"ROUGE-L Score: {eval_scores['rouge_l']:.4f}")
    print(f"BERTScore F1: {eval_scores['bert_score_f1']:.4f}")
else:
    print(f"Mismatch: {len(generated_commentaries)} generated vs {len(reference_commentaries)} reference")


Error in batch BERTScore: There was a specific connection error when trying to load roberta-large:
401 Client Error: Unauthorized for url: https://huggingface.co/roberta-large/resolve/main/config.json (Request ID: Root=1-690ec37f-1010dd5f3c69eb613f6f9f42;0a701e02-89dc-4e0f-b017-b916aa26f09a)

Invalid credentials in Authorization header, falling back to individual calculation
Error calculating BERTScore: There was a specific connection error when trying to load roberta-large:
401 Client Error: Unauthorized for url: https://huggingface.co/roberta-large/resolve/main/config.json (Request ID: Root=1-690ec37f-746c5a1f58c2f8515bf3e951;443edc4f-80da-4c0d-aa61-08dac74e14fb)

Invalid credentials in Authorization header
Error calculating BERTScore: There was a specific connection error when trying to load roberta-large:
401 Client Error: Unauthorized for url: https://huggingface.co/roberta-large/resolve/main/config.json (Request ID: Root=1-690ec37f-578c4f091a9521cc2f52c94d;8aedadbd-c968-4162-b8a4

In [ ]:
# Export results to JSON
def export_game_to_json(game: GameData, quality_metrics: List[CommentaryMetrics], 
                       filename: str = "commentary_output.json"):
    """Export game with commentary to JSON."""
    output = {
        'game_id': game.game_id,
        'white_player': game.white_player,
        'black_player': game.black_player,
        'result': game.result,
        'metadata': game.metadata,
        'moves': []
    }
    
    for move_data, metric in zip(game.moves, quality_metrics):
        output['moves'].append({
            'move_number': move_data.move_number,
            'player': move_data.player,
            'san': move_data.san,
            'fen': move_data.fen,
            'evaluation': move_data.evaluation,
            'depth': move_data.depth,
            'commentary': move_data.commentary,
            'quality_metrics': {
                'clarity': metric.clarity,
                'relevance': metric.relevance,
                'factual_accuracy': metric.factual_accuracy
            }
        })
    
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(output, f, indent=2, ensure_ascii=False)
    
    print(f"Exported to {filename}")

# Export the processed game
if game:
    export_game_to_json(processed_game, quality_metrics, "commentary_output.json")


Exported to commentary_output.json


In [ ]:
# Clean up
system.close()


✓ GPU memory cleared
